# Inspect the new response trace

This notebook runs the SDK against local Narada services and displays both the existing trace fields and the new OpenAI-shaped flat trace. Start the backend and frontend first, then launch this notebook with:

```sh
uv run --with jupyter jupyter lab
```

In [ ]:
from __future__ import annotations

import os
from collections import defaultdict
from typing import Any

from IPython.display import JSON, display
from narada import Agent, AgentKind, BrowserConfig, BrowserEnvironment, Span, Trace

## Configure the local services

The defaults expect the backend on port 8000 and the frontend on port 3000. `NARADA_API_KEY` must be available in the notebook process.

In [ ]:
os.environ.setdefault("NARADA_API_BASE_URL", "http://localhost:8000/fast/v2")
initialization_url = os.getenv(
    "NARADA_INITIALIZATION_URL",
    "http://localhost:3000/initialize",
)

environment = BrowserEnvironment(
    config=BrowserConfig(initialization_url=initialization_url),
)
await environment.start()

## Display helpers

The API returns a flat list. The tree below is reconstructed entirely from each span's `parent_id`.

In [ ]:
def exported_trace(records: list[Trace | Span[Any]] | None) -> list[dict[str, Any]]:
    return [record.model_dump(mode="json", by_alias=True) for record in records or []]


def span_label(span: Span[Any]) -> str:
    data = span.span_data
    return (
        getattr(data, "workflow_name", None)
        or getattr(data, "name", None)
        or getattr(data, "message", None)
        or data.type
    )


def show_trace(records: list[Trace | Span[Any]] | None) -> None:
    if not records:
        print("No response trace was available.")
        return

    trace = next(record for record in records if isinstance(record, Trace))
    spans = [record for record in records if isinstance(record, Span)]
    children: dict[str | None, list[Span[Any]]] = defaultdict(list)
    for span in spans:
        children[span.parent_id].append(span)

    print(f"{trace.name} ({trace.trace_id})")

    def print_children(parent_id: str | None, depth: int) -> None:
        for span in children[parent_id]:
            print(f"{'  ' * depth}- {span.span_data.type}: {span_label(span)}")
            print_children(span.span_id, depth + 1)

    print_children(None, 1)
    display(JSON(exported_trace(records), expanded=False))

## Direct Operator run

A direct call should produce one agent span with user-facing action spans beneath it.

In [ ]:
operator = Agent(environment=environment, kind=AgentKind.OPERATOR)
operator_response = await operator.run("Open example.com and describe the page title.")

print("Legacy action trace:")
display(operator_response.action_trace)
print("New response trace:")
show_trace(operator_response.trace)

## Agent Studio GUI workflow

Set `NARADA_DEMO_WORKFLOW` to an Agent Studio path such as `/owner/workflow-name`. A useful demo workflow contains an agent step, a loop, and a nested custom workflow.

In [ ]:
workflow_path = os.getenv("NARADA_DEMO_WORKFLOW")
if workflow_path:
    workflow = Agent(environment=environment, kind=workflow_path)
    workflow_response = await workflow.run("Run the observability demo.")

    print("Legacy workflow trace:")
    display(JSON(workflow_response.workflow_trace or {}, expanded=False))
    print("New response trace:")
    show_trace(workflow_response.trace)
else:
    print("Set NARADA_DEMO_WORKFLOW to run the workflow example.")

## Cleanup

In [ ]:
await environment.close()